# Would you trust this model?
**Imaging & CNNs workshop · Laptop part 3 · teams of 2 to 4 · about 22 minutes**

## The situation

A hospital wants a model that looks at a chest X-ray and reports which of 14 findings it shows. Someone asked an AI assistant to write the training code, and the code further down is what came back.

The code runs without any errors. At the end it reports that the model is correct about **95%** of the time on the test set, and it says that the model is ready for the hospital. The hospital would like to start using it next week, but first they have asked your team to review the code.

## What you are looking for

The code contains **four problems**. None of them is a typo, and none of them makes the code crash. Each one is a *decision* that was made badly. Every problem either makes the 95% look better than the model really is, or makes the model itself worse. You have seen all four ideas earlier today.

## How this part works

| Phase | What you do | Time |
|---|---|---|
| 1. Review without AI | Read the code as a team and fill in the worksheet. Please don't use AI tools yet. We want to find out how much you can spot with what you learned today. | 12 min |
| 2. Review with AI | Ask Claude or ChatGPT to review the same code, and compare its answers with yours. | 8 min |
| 3. Verdict | Decide as a team whether the hospital can use this model. | 2 min |

You don't have to run the code to find the problems, because reading it is the exercise. If you want to test a suspicion, you can run it: training takes about two minutes on a GPU.

💬 **A good way to review as a team:** one person reads the code out loud, step by step, and explains in their own words what each line does. The others interrupt as soon as something sounds odd.

In [ ]:
#@title ▶️ Run this cell first. It loads the X-rays and two helper functions (about 1 minute).
!pip -q install medmnist
import copy, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from medmnist import ChestMNIST

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load(split):
    ds = ChestMNIST(split=split, download=True, size=28, root='.')
    x = torch.tensor(ds.imgs, dtype=torch.float32).unsqueeze(1) / 255
    return x, torch.tensor(ds.labels, dtype=torch.float32)

# The dataset comes with its own split into training, validation and test X-rays.
train_x, train_y = load('train')
val_x, val_y = load('val')
test_x, test_y = load('test')
mean, std = train_x.mean(), train_x.std()
train_x, val_x, test_x = [(x - mean) / std for x in (train_x, val_x, test_x)]

def train_one_epoch(model, X, Y, loss_fn, optimizer, batch_size=128):
    # Show the model every X-ray in X once, in random batches of 128, and update the parameters after each batch.
    model.train()
    order = torch.randperm(len(X))
    for i in range(0, len(X), batch_size):
        batch = order[i:i + batch_size]
        loss = loss_fn(model(X[batch].to(device)), Y[batch].to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

@torch.no_grad()
def predict(model, X):
    # Return the model's 14 outputs for every X-ray in X, as one table with a row per X-ray.
    model.eval()
    return torch.cat([model(X[i:i + 2048].to(device)).cpu() for i in range(0, len(X), 2048)])

print(f'✅ Ready. The dataset has {len(train_x):,} training, {len(val_x):,} validation and {len(test_x):,} test X-rays.')

---
## The code the AI wrote

The code has four steps. It uses the two helper functions from the setup cell:

- `train_one_epoch(model, X, Y, ...)` shows the model every X-ray in `X` once and adjusts the model's parameters as it goes.
- `predict(model, X)` returns the model's 14 outputs for every X-ray in `X`.

`X` always holds X-ray images, and `Y` holds the matching labels. Each label is a row of 14 zeros and ones, with a 1 for every finding a radiologist saw on that X-ray.

In [ ]:
# ===== Step 1: split the data =====
# Put all the X-rays together, then make a random split: 80% for training and 20% for testing.
X = torch.cat([train_x, val_x, test_x])
Y = torch.cat([train_y, val_y, test_y])
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

# ===== Step 2: build the model =====
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 14),
    nn.Softmax(dim=1),  # turn the 14 outputs into probabilities
).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ===== Step 3: decide how to measure quality =====
def accuracy(model, X, Y):
    predictions = predict(model, X) > 0.5  # a finding counts as present when its probability is above 0.5
    return (predictions == Y).float().mean().item()

# ===== Step 4: train for 5 epochs and keep the best model =====
best_accuracy = 0
for epoch in range(5):
    train_one_epoch(model, X_train, Y_train, loss_fn, optimizer)
    acc = accuracy(model, X_test, Y_test)
    print(f"Epoch {epoch + 1}: test accuracy {acc:.1%}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = copy.deepcopy(model)

print(f"Final result: {best_accuracy:.1%} accuracy on the test set. The model is ready for the hospital.")

---
## Phase 1: review without AI (12 minutes)

Double-click this cell to type in it. Write one row for each problem you find. The most important column is **what would go wrong in the hospital** if nobody fixed the problem.

| # | Where (step and line) | What is wrong | What would go wrong in the hospital | How you would fix it |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |

<details><summary><b>Stuck? Click here for four questions that every reviewer should ask.</b></summary>

1. **Where does the test data come from?** Could the model have seen something from the test set during training, even indirectly? Think about who the X-rays belong to.
2. **Does the last layer fit the task?** How many findings can one X-ray have at the same time? Can the last layer of this model express that?
3. **Would a lazy model also get a high score?** Imagine a model that ignores the image and always answers "no finding". What score would it get with this way of measuring?
4. **How often is the test set used?** The test set is meant to be a fair final exam. Is it used only once, at the end?
</details>

### Extra challenge: prove it with one line of code

Write one line of code that shows the 95% doesn't tell us much. The setup cell has already loaded `test_y`, the true labels of the official test X-rays, so you can run your line without training anything.

In [ ]:
# Your one line goes here.
# Hint: what accuracy would a model get if it said "no finding" for every finding on every X-ray?

---
## Phase 2: review with AI (8 minutes)

Now compare your review with an AI review. Copy the code cell above into Claude or ChatGPT and ask it to review the code. You decide how to ask, because the way you phrase the question is part of the exercise. Try at least two different questions, for example a short one ("Review this code.") and one that explains what the code is for and who will use the model.

Double-click to fill in this table.

| Question | Your answer |
|---|---|
| Which problems did the AI find that your team missed? | |
| Which problems did your team find that the AI missed, or explained wrongly? | |
| Did the AI say something with confidence that was wrong, or that didn't matter? | |
| Which way of asking gave the most useful review? | |

💬 **Discuss as a team:** suppose an AI had written this code for you, and you had asked another AI to check it. Would the problems have been caught? Which of the four problems could you only find because you know how hospital data is collected?

---
## Phase 3: your verdict (2 minutes)
Choose your answer in the form on the right and run the cell. Be ready to explain your choice to the room in one or two sentences.

In [ ]:
#@title 🗳️ Your team's verdict
verdict = "Fix it first, then test again"  #@param ["Use it as it is", "Fix it first, then test again", "Start over"]
the_worst_problem = ""  #@param {type:"string"}
one_sentence_for_the_hospital = ""  #@param {type:"string"}
print(f'Verdict: {verdict}\nThe worst problem: {the_worst_problem}\nWhat we tell the hospital: {one_sentence_for_the_hospital}')

---
## Solution
The instructor gives out the password at the end of this part. Try the review yourselves first: you learn most from the problems you find on your own.

In [ ]:
#@title 🔑 Show the solution (the instructor gives you the password at the end)
password = ""  #@param {type:"string"}
import base64, hashlib
from IPython.display import Markdown, display
BOX = {"salt": "617242e4a8572f77", "check": "9b33684c26c0155d4d93e6b5b501c690f0f7770793561356c3d7d1ebc255d481", "data": "8pDkXPcdk2yQ3oAsw21BFHKGpV4acamq2i+AmDU/VBC6D+ntGe/E9Qk9FqE3ncaNlUNHHor/heagKvX1gNXOrSyb/Kkjgk5am/o0EL/m5LbQOs9P6TOZDqudCrq7LlyoQo8VlbTxupbSIO3/DnghEUYnbkJbVzrS7XWItQEn2S7z4vYCBzv6ekWD2PjMhmarLQT6IDp5rLIgKexhp93K4w47N/wYoFoWa53KIjIBBHF2c6FzxhB1nnx7FdgHpAi9XhTDNx/Wp93l/q5Y3b1J8j5kwo5jfKKB64MZjUb4tzO7OWJ5UCFtPpFCwDd7IKwVOrQCIFG12vfZW15CM4nU6ZQCHgjEi/2WFqfw9hAr9g3ZYsGrqQjVSU2XgzN3mf6vfuc1jKb4cmuOxHcZ2imVb3z1IDdIjj/8pCtd/QusFk1HDOWq2WqZfpTCZzVUMl2L2zJIwWhhbbHzmmLzQUShjfFZJX/IzNSsPtysjaLx/t1k39M+6i2C8L+RrqAKZr1ypDbbvCyre1mhNZskXu2+49qa9C6k/4fCbo9Ri5+UHTTsQ/Xv7O+4RHsRB5Huc/GPWI4G9mE7H2wA4fUT5vzfe/fHyw25kvuwJ38afG74seAt9NpGa9rvi6KktEyJrxJEaT76/ZsvLHOFsK0CDDMSogADahFXn2bAP1GFxupZzAUYbgq7xQ8HyXZ7X2bU6tksFIheKfLWtOECEv+dQu2v4Ox2Ku+t1fbfBm0hCEnjtry4JPjLXKYvbvaExGvlK6bVqLbdmEzrCYonm9yOcyYwwSg7SqClIz8xzC8pjapRIfoQ6Vi9g5rlH0VtNDiLnHyCVhILYnc4z5/hHmv1cpU71AVmtFw/pLBOsLCgTQuuNam2KcD3tkarMimz6jv71XQsUGq4rBSGCGnDsaauYvhzB3VTdZZIy0Jz3Q2Tv3/Gjym1YSJIszgz39Lp244+lH9u6Hq2svkEegoqGtmFtGgDYAg0yWaRL8CoiuXKIi67EBz0d3okueWDphpGS8Ocha+wJFf25/6E2lOtI5YEOyarv+RzyxandS1qG+lik72YYnGi6ezUxGEfVWXb2MPw+s1JBIkRVm2JknK952v2QGza+AbeO6HFY0L5wLzDeAmAp1VqUlF8KxTdATae5ow66JRvSAAOapyrXcuXrcrukw7Qk+ZmJ/3fQls8x1lJL46GY/VykI2G0EA6Ww5Q3Z1KfYSHG9YLv/ot710b97r+WcOQQvNh4KUoc3Uvjr8/w80Ai8xuCOOBhF+64Ql9LVhgVOvAmioEa3nx0rrDNbrkyr5OEzB+cyXatfBPWrlpJIwQtSUlh7ZckII8wkAAAy25LQnFocTOVAENlVYwQs40xZI3SYPiSyjaII4Yzw7/NN01jlnfvKPYCTfDUN1POHdzUmtaGzXPJ/xZ2CVheS9xeMYVABV7NnLEMqYzQ9V3CEJNoriifObFDrAPMJ7WW1yZxZAhp+2nuoGnsLSqB84pmR7lEWj8YQ/VfYRmntMUSAkB7DYUa91WfETu/XvTPy5dn2NrWcfsMqeU2nHv1A4URSnP41BZiWcezvAsTZ399rFrjhyEoCoS96TA3HkO5kKhzaHxn+OCovfEvnfBkIEE8EtwhUv5Ggmuqe6+je67owlefnkqiFW8z+bWeJrBYcB9p8DYEvwCEpPgOndhOJ4aqhMu5T4bRX+X1MZohlVbM0U1uy/9yG9cOfxK6AcB8b8sCu568Z1lzeR/0oYVXzOFSOpv935Hdo6XyH2mzq/OQskH+QpePMD6aQO+kTtuKYNrd8DSBONncR03M5ka2xlQsphzgdzCb0QkYZX9HO4mkdxwqXhVi/h/I20uWeid3e1zKe8QxRwGoFAywAPjx75gNrrp7LYYacokhCstgPRVR+qf32F0vcXZbrl8U0yygmrgavw0Vdks40UB8w+neVjIC2B+7DAPGCW+rNdqGhB0jSRtI8+B8zdDWVXc6pFPowcfP8754gO5qQyoOyOsNJVnF2Uo+D7NzlCKUC2zZWh3Zl4m8u863mJv1XFwa0k420qZwvoSg7yIYhp3E768CfTs2GwQ3XF1s9Lpf4y33ZMoUgdYq+SsV0Cp71qeJ1QxH98qsQ39Svi6dK5uFsMoUd/NznQxowsbJY0TVaEP8VJB15QKMFuUWhvsuVSkZnjM1OlMvc4JAYuDJHX5vOaZKBko6IZOLL+FfMRqsq2+qIzL5eyCC30wGNaYvMsfyhh/oN6GbTxKtmk9Tb8o81UiRM5Z4AFxN0p2LNQvBE4+3sYdoa1t6qc7mP4WRoYf91dueXJ+NzE8+p1kp+lNPGkEfPYOrfEOu4vGuncBXigCQXcxEFyow26oMdkkGEHgkn5FkeyU2G/09374hPamiyaFJhOe3Q4g3aQlC8DkrDK7t1wQsOYWU5d+b57SQfALHpNsHydkrCjg39+D3wv3IjaVomlaJFE/QNHJdZTVxUhZI8GcNKpm6VB7anK13HJOgscF24ky2aNZCUBEB5ch6ywt5wPMKLzod2UqVrUoqCrx7hUKamIziAsMSG40A9htBuuw+7eR0YcQDALpgnsfMqpN1+5bzSMAtGqWb0XeJSCb/k88ZSjx27CSxfHadwU4QjNQzVpY8hmRE7pyl0YyR7PNgTiFGZS2nX8FpqRAGsTiEOZ+FjY6x0yCebNowgstohdk3H0+018LTvWJhOQkrE071BWZqeGHQWPxoLFw9HhAkc2wxYED0y+XeBU3E4tUpdJNqTluhG9ISLyK9+ARLM1UOZvPXIwiBGrmxEBazSPjU/IUB83nUDl8Jm42GdnFmT5kygZuT+YputXmhyTSsEIiN4WIw7cmWuOaR+9iB5jgpzNVJ8CR767y/uZqqLQYkJgIRZboU94Y1eFRIZGZgdUpwernOVkUlcUWc3OluaYSCFsgLug081rcWf5MXY6cAoqwabzVV+Uze2fr8fw5TMw9rX9ZMxqHUOIzt7d7om59LnSgH54YgWI873C2YbZQkeoEFT7RgqV4uDHB5D888XsIvz3/PIyUqIYUj53UYBSY9D2KSCxZOn5cgNpb5VsmJB8w1B8uGSI3Obo93QEMRa2Wtly2Xit2crsFbYOw1n4gPHOVyXTbKpgFUPJ/ozbfS50p+xRC68kKcjUWgZPM2fJd75mZiOxmOVmueHlNxgaVWat8UMrEIwspIaZmQw5CdIr+ITEuawzuGGdMdoXn6JqI65Tulz9Ck0Mjw4rzZfhuxY9+FqFwb8zV+AXSorHxGoVCKukVsPbGCZXwoXY6lFkqbzf1/ZOEeeWlIPiFi+o4LE8geertf2mt+wKqGGfVhjPP7B4p0Pdwm27LdAMjt6llt80aOvRdZdtFHjtkglsoT5l43KG3W0EkYktg8FAmd6zW0qEOyduX28sF/TY3Euv1Ldtq4NQ8bLMLZYmZIzpjZOJxtjW67Jzuam4Yk7a8HlAX+d13UDSMOXRvhRX5Aayb9ynrHQaDl86DupK5umKEzfnviG3YjRC9+BrMyilstphhas9oMYJ0Fxt/Jgsahjsemo0SlnyC2MsK9DnCh0ig1ael8TTZIkxjWVkhjcfIohVKyf95nBPyGm0bMH1Sj5I68jknnhG+QHt6XZ7vjjOUfE8a4InQnRM2lCTELD1zjSI+41R9hB3ptCsxjD/trKa8tEVSYIHevq43SujyOQOcrUXAhOWzw24KR8nX10CgYsQC46WKbE2XF4+9eSly71Mzyqo/iwLy3bTihxyiaIpziRtNDhG7E4L7QzcZJNZq1dZgWaLUzvwJXrQUCyNpLN0X+k3X4EklgHS4rb5GYbLgW6fL4FjPm1xPPPKtPKAq3KL8Cl7GDb3jK7KvcSIr0+cRvV8axVX6Ht68pFhRG61rSlrNF2Pb9+C8KOdm7c1hj17n/SkWYr67XxqdRWjGgm4ZzCBelfELpXwunOpx6DoMH21kpRCdesHHUbK3MdjfaoYTsFvn/JTJKEIlxa5yOE3pvj+rmnjdge3HxCvGCOjpyC2evjafmV2ZxD8cfTnqnmcgFND+VXh+CdpztqtTcT0K9EQRBeTc+fXds5IBOdAdXut1sioUUnitxHF7aYOv0Mgc7Txtm/WESq4m0y8V2tDYrBQ9G/9mKRLw4fNrgtzLxm0Kmu7z7vZkwc04zzwzc9gEHCPLU2Jp0A9AjIjEC/a8PjavSTwnjAJOg6m3gJBetk18HOwO/w1+UfBMZuEEknkz7ywtZQf/zsB7uBo6x5mOS6C2Sdlw6Nh7s9T1JIWakXfk76fsdRo7ADQBsWPrFI0CphAMipd2P0rl1viJJD77GIDuuGyYtU7WTo/8JrSHC0mQX1XgiPdlGi3u1atxQjKSqTPyUzn2bf6m1p/Y9q0MPDLc/isQ/kSdpCkofflEC0NLWO2MX3NB68Yye2v49YDSraGD5IY6RUmkk7sptfXZKY9gl8D4hnH4TDgkrvGyZBpQOgedUoWhZphSL0lCabD3ElSEw1qWm86Ip1+qEj01kwdMhVGWpG+eb2n4Od98SKkQ4scgFkWA2HZVkcoEn+rWZUGTSjfIpQVL6JUdEtohhX8myisjQYJlVnV6UAvNk1uXmPduwU7H6zpY+sOya0wE9Frhs+CA8wK387bNoH6c/XUxxC/KUu9Qc6TgrY5uo+jdSbTe/99D1gYZaKUMqmqqJS3wYHIkSf3pSN1fyan0KDMqQdL5vyhuVYC4IkDYXaslWb1llwcBDd9Zf+WJCdnP7Vy7TRXSK7NmKidmu87B0vBPUb6iXeRZMNv4AwnGSJ2/9MloPVN6csr6rDsKEyk2kE+Mf4eYSnHT4BhvpPycUeBnYVDagj7AiiUvS+P4qkLtUxhq6PCOHgO70Rn4engg5MvvztabhHZw+iZ8hHq7ftfpuElWam6QE2i9xmm9F6VmRi1c2Vv2Ubv9Aon2kVcQu1mm+6E2tay96gwUYxGoqqFZkh2exLGCDZRL6q8cpUlS8CDCjaRKIFrmDpGuc+c26eZk2FyilabSyVr9L/4UlwQtWSMDNfAWEcAgr3ZF68JELPTi45JjQ+ciI1puvAiMFvA/iTg7wvoFEGHcD6j+LgDisEUwF5+ArXw/+Il+YEHZsHjF0Cfo6CI/M7rFslkOFBv8mzIPy1vGlJdjKMDyMFOMORaB7NnDZQvcjyBz4q2oP9MDLkLxcBbvV3tkCQzIE/MLKNXqjvBnBntqDAVpIZkFSWfCgcedoqZuEh1dzhZHOCooZlPEQ1KW/j9pBqlv+kIrwSY5XDuHnnJKow1UP1gn95eVVwWc9nXENoLOwt7ep8tHL4Uc9SL6ecZZ8mkPTHGKwDcEht7GFiTRwMVjdtO50Kh8HkuLJwOepGp/rKAPtbUv06TojepDcB5j01Tfc44iQX9Sy77kRos2ewy0nkP3UY9ovHN0+ujjVsKKB1+kr2cN1CRsqn4IryPS6r2WkcEtJYSB0JMUqgb4UEUu/m2A5DlYg5m0z+axESFljWg/g4cg0q57GkCquA4q9tL9jF7w2256fFWiSeqbRf+mfxm5gAGtTpuXLrQ7mMBqAIPq8iY/s87FVQgSamdze2quO8U30wVMKLmgrr9wCAmtlL+L6R/WveVfQaI7OnGT9tyL41br49nh+0UNz3VpWcCMW9FLSK/ZpPwQW2HlBTDXBe1AbtNTOCR6krJKnzlt1T5cD9DQEDDOm0QRO4nqJ0vtUi4o+D41jU/9i8bI0sck4S/OfwrFVoPvX/wpflqpy6NES+UL8b6acl2bBmu1d5yNBk5v5AXRnVWY6mDINpYPJJvKSW4HQOiWBNOQ019M/Y63iy2WMZVPOywR2OjQZ0pfu6WfQ5OBDmbjmulmAxlUpa+TbYZT0NRalLBJFPjYs/VkeWpFU8sO+BeBPgLyVyMmIGlEo1byTGTJ7PsmafIl0jEVro0tnyo6uCTML4KD7faHJTrhwbF6S5iuc/N7+PyeKMjGR236SoLj07A4e52YP1eDEXoSzX0OXEcAFcF/i+s/FZEkp5G3dUz+X/cpKZWzw8Fui0Uvl2m5dLlACZifmdyZpSClFjkskNZRrU+Ld/RGJ+Tm2ksAgoGMJwmI/i4TpbhEBymauYP0QXEiYHAup2sNMDmPNepnS/fmf9a1qpSUCWOM7Dn6gNDIw2+b0GFMdLYtsXeGcZcB1AHuYWjsRmFIPbaz3pWWa7ygM2ciRdtlEh0YlK9COHN8b2GgSaS6fv+tRPDniqKia1aOK2FbVP1UN6LU/6s+VywV5DJ2JP3wCsGzlr/WWagG+FxVntnJw0swNTRI/J8A2JUdDmiDaNxgOUL04v5zUV3blJxynSBRlB9RUM80Ob1dvGJOaup7E1cqMx+0gCGlAhwz+RcuXFLeYQsj1Pz+tgF0QePg4qcB4gFb4rhPT8F/QqV0veE0jXFTgZpNZBoYPD8sK6Wq1KLL9LR4ErKQQ6jUZXaKpbZNGGEUnrs0zbI6/wlEagq3COyZC7PZakCQ9oDqTe6m6HwUcQR0qWfsam2XrUtcMtF/G+OUlUmT4TKKwwoGdM/wyuQYyGRUmJ0VYLPe2Tei39iUXhWnQYDzetp99qUF4nFxCKcFSFyF/RRNbeek26Pn5H92gjJC94HP1MeByy7aM3Zzmwbn58/oYrHiHPwYnC3hVowkjtyqZx28+mgEwPaQYKrl/hJ5Nxp1lSl6jnjqsnSM2qttJRoRVlTJEikJe2SuxLyDnhDEEJq/uWqcR/VN8zhSDugWmvta8tqxEpSUm6j3tUMKT9TzzJXQgcQYjFCsS0sN/tJKKR06FQfhHUULmJUzO1eFIORB/BeESy9+gNBbOG4KBE3Z6KJxIzDcDzqScblFoZoKzJU8tjLdRHx6oH9q7munSbraKYKZbVYFlY9B9AW/GoUOMOWL0Qup/O0Y/D/ZdL7opUBVsd77Sd4bZryThdPQ1NrnXSwwRx1of5nOSh3PrV4+RvIAZYDB9XLCnciSr67kUyYOVoJkLsaXVz2ZdJmt1rgCiFOqGj78CYxZ6CMzhPXfuwAdnKAnkxqajj4kKcJTOwG7568sUTMq8VqpPBPF6g9uZU4Ga360HRwA6vXqGn8Dc95tCiL6mPavCSS4Hr58t28BP/GF9AY0gq5rJe36ZI+HGmw6d0BUq9EO8uKpSU2ZUOL932Si3d8j6Kh1hv9jYvFASt7oUc9zE3Er5GwZ6Aah6arr31sA7lEatnzta8CKeFZv+xhuqdp2KMt7zetDpaaiGKJXhY47+K+h+sUjJyDhpv3+1VjI+NITx1AlmoRglFYF//P3CB1yeqvxQVG7WWMaDUlpz4Ha0lFpPYISZDJhNv4WxcICN//w8VWsRmdWS/rRY2zd4XbdiivOnTVl2yGdbOCKLpawxCwOelvMddEdmuGp3+x68xbz4uTq3vQk5iYTUUnCaoPZGtnGnxJIFD2Zm5FFydhgBrvNV2LGM64n0pKKoY88OcZfGQ5bljEIwpKMw3m45FtdVFh+GLsmOLnBUdHxYblAoCgtI5JEJOoWZVkz4+BJrXT6ENipo6xIiiMAhg/P838NjF2f/aBGnN4b8WbNRi6w905gwvNt4FbkV9k8BHaKc3XdXUxTXMJCOslwMuWIoydPfjosOrifSuD8yAFIUzGQp8Hjd58izdydJiiJXJSXP74xpx1teChjontV7iNMSJWRA7eVDI56pgwq2W/pyGapfRrtEydww7YNHpFnKR08IeWdVO9QyzwD4GeRp2pomt1SuvADp+uYWaynNLlvFV/Sl+DCag=="}

def _keystream(salt, pw, n):
    out, i = b"", 0
    while len(out) < n:
        out += hashlib.sha256(f"{salt}:{pw}:{i}".encode()).digest()
        i += 1
    return out[:n]

pw = password.strip().lower()
if not pw:
    print("Type the password in the field on the right and run this cell again.")
elif hashlib.sha256(f"{BOX['salt']}:check:{pw}".encode()).hexdigest() != BOX["check"]:
    print("That is not the right password. Check the spelling: it is all lowercase, with dashes.")
else:
    cipher = base64.b64decode(BOX["data"])
    text = bytes(a ^ b for a, b in zip(cipher, _keystream(BOX["salt"], pw, len(cipher)))).decode("utf-8")
    display(Markdown(text.replace("__CONFIG__", globals().get("CONFIG_ID", "?"))))